# 深度學習原理與框架

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明人工神經元如何透過「輸入、權重、偏置、激活函數」產生輸出。
2. 理解感知器只能處理線性可分問題，並觀察 XOR 問題為何需要多層網路。
3. 比較 Step、Sigmoid、Tanh、ReLU 等常見激活函數的輸出特性。
4. 用 NumPy 實作前向傳播與簡化版反向傳播。
5. 使用 sklearn 的 MLPClassifier 體驗深度學習框架如何封裝訓練流程。

本練習採用輕量資料與 NumPy / sklearn 示範核心概念，避免依賴大型 GPU 或深度學習框架。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立固定亂數種子，讓每次執行結果較容易重現。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)

print('NumPy version:', np.__version__)
print('環境設定完成，可以開始深度學習概念實作。')


## 核心概念說明

### 1. 人工神經元

人工神經元會接收多個輸入值，將每個輸入乘上對應權重，加總後再加上偏置，形成加權和：

`z = w1*x1 + w2*x2 + ... + wn*xn + b`

接著，神經元會把 `z` 丟進激活函數，產生最後輸出：

`output = activation(z)`

### 2. 感知器

感知器是早期且簡化的人工神經元模型，常使用 Step Function 作為激活函數。它適合處理線性可分的二元分類問題，例如 AND、OR，但無法單獨解決 XOR 這種非線性可分問題。

### 3. 激活函數

激活函數的關鍵價值是引入非線性。若神經網路沒有非線性激活函數，即使堆疊很多層，整體仍只等價於一個線性模型。

常見激活函數包括：

- Step：輸出 0 或 1，常用於感知器概念示範。
- Sigmoid：輸出介於 0 到 1，常被解讀為機率。
- Tanh：輸出介於 -1 到 1。
- ReLU：負值輸出 0，正值原樣輸出，是深度學習中常見選擇。

### 4. 前向傳播與反向傳播

前向傳播負責產生預測；反向傳播負責根據損失函數計算梯度，進一步更新權重與偏置，使下一次預測更準確。


In [ ]:
# ── 示範：人工神經元與感知器 ────────────────────────────
# 這段程式碼用 NumPy 實作單一人工神經元，並示範感知器如何學會 AND 邏輯閘。

import numpy as np
import pandas as pd

np.random.seed(42)

# AND 邏輯閘資料
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y = np.array([0, 0, 0, 1])

# 初始化感知器參數
weights = np.zeros(X.shape[1])
bias = 0.0
learning_rate = 0.1

def step_function(z):
    return 1 if z >= 0 else 0

# 感知器訓練
for epoch in range(10):
    errors = 0
    for xi, target in zip(X, y):
        z = np.dot(xi, weights) + bias
        prediction = step_function(z)
        update = learning_rate * (target - prediction)
        weights += update * xi
        bias += update
        errors += int(update != 0)
    print(f'Epoch {epoch + 1:02d} | errors = {errors} | weights = {weights} | bias = {bias:.2f}')

# 測試結果
rows = []
for xi in X:
    z = np.dot(xi, weights) + bias
    pred = step_function(z)
    rows.append({'x1': xi[0], 'x2': xi[1], 'weighted_sum': round(z, 2), 'prediction': pred})

result = pd.DataFrame(rows)
print('\nAND 感知器預測結果：')
print(result)


In [ ]:
# ── 示範：激活函數視覺化 ──────────────────────────────
# 這段程式碼比較 Step、Sigmoid、Tanh、ReLU 的輸出曲線，觀察不同激活函數如何改變神經元輸出。

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 400)

step = np.where(x >= 0, 1, 0)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)

plt.figure(figsize=(10, 6))
plt.plot(x, step, label='Step')
plt.plot(x, sigmoid, label='Sigmoid')
plt.plot(x, tanh, label='Tanh')
plt.plot(x, relu, label='ReLU')
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('常見激活函數比較')
plt.xlabel('輸入 z')
plt.ylabel('輸出 activation(z)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('觀察重點：ReLU 會保留正值並將負值變成 0；Sigmoid 會把輸出壓縮到 0 到 1。')


In [ ]:
# ── 示範：前向傳播與反向傳播 ────────────────────────────
# 這段程式碼用 NumPy 建立一個極簡單的神經元，透過前向傳播產生預測，並用梯度下降更新權重與偏置。

import numpy as np

np.random.seed(42)

# 簡單二元分類資料：輸入越大，越可能屬於類別 1
X = np.array([[0.1], [0.3], [0.5], [0.7], [0.9]])
y = np.array([[0], [0], [1], [1], [1]])

w = np.random.randn(1, 1) * 0.1
b = np.zeros((1, 1))
learning_rate = 0.8

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-9
    return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

for epoch in range(1200):
    # 前向傳播
    z = X @ w + b
    y_pred = sigmoid(z)
    loss = binary_cross_entropy(y, y_pred)

    # 反向傳播：對 w 與 b 求梯度
    dz = y_pred - y
    dw = X.T @ dz / len(X)
    db = np.mean(dz, axis=0, keepdims=True)

    # 參數更新
    w -= learning_rate * dw
    b -= learning_rate * db

    if epoch in [0, 1, 2, 10, 100, 1199]:
        print(f'Epoch {epoch:04d} | loss = {loss:.4f} | w = {w[0, 0]:.4f} | b = {b[0, 0]:.4f}')

final_pred = sigmoid(X @ w + b)
print('\n最終預測機率：')
print(np.round(final_pred, 3).ravel())
print('最終分類結果：')
print((final_pred >= 0.5).astype(int).ravel())


In [ ]:
# ── 實際應用：使用 sklearn MLPClassifier ───────────
# 這段程式碼使用 sklearn 的多層感知器分類器處理非線性資料，體驗深度學習框架如何封裝前向傳播、反向傳播與參數更新。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)

X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
model.fit(X_train_scaled, y_train)

pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)

print(f'測試集準確率：{acc:.3f}')
print('混淆矩陣：')
print(cm)

plt.figure(figsize=(7, 5))
plt.scatter(
    X_test[:, 0],
    X_test[:, 1],
    c=pred,
    cmap='coolwarm',
    edgecolor='black',
    alpha=0.8,
    label='預測類別'
)
plt.title('MLPClassifier 在 make_moons 測試資料上的分類結果')
plt.xlabel('特徵 1')
plt.ylabel('特徵 2')
plt.grid(True, alpha=0.3)
plt.show()

print('觀察重點：多層感知器能透過隱藏層與非線性激活函數，學習 make_moons 這類非線性分類邊界。')
